In [14]:
import pandas as pd
import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, log_loss
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.calibration import LabelEncoder

In [21]:
df = pd.read_csv('../Cases/Wisconsin/BreastCancer.csv')
le = LabelEncoder()
df['Class'] = le.fit_transform(df['Class'])
x,y = df.drop('Class', axis = 1), df['Class']

ohe = OneHotEncoder(drop = 'first', sparse_output=False)
col_trnf = ColumnTransformer([
                                ('OHE', ohe, make_column_selector(dtype_include=object))],
                              remainder='passthrough', verbose_feature_names_out=False)

col_trnf = col_trnf.set_output(transform='pandas')
x = col_trnf.fit_transform(x)

In [22]:
x_train, x_test, y_train, y_test = train_test_split(x,y, random_state=25, test_size=0.3, stratify=y)
scl_x = MinMaxScaler()
x_train_scl = scl_x.fit_transform(x_train)
x_test_scl = scl_x.transform(x_test)

In [23]:
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')


etas = [0.001, 0.01,0.1 ,0.2,0.4]
lr_sch = ['constant', 'optimal', 'adaptive', 'invscaling']
scores = []

for e in tqdm(etas):
    for lr in lr_sch:
        sgd = SGDClassifier(random_state=25, eta0=e, learning_rate=lr, loss='log_loss')
        sgd.fit(x_train_scl,y_train)
        y_pred_proba = sgd.predict_proba(x_test_scl)
        # y_pred = scl_y.inverse_transform(y_pred_scl.reshape(-1, 1))
        scores.append([e, lr,log_loss(y_test,y_pred_proba)])

df_scores = pd.DataFrame(data = scores, columns=['Etas', 'Learing Rate(Scheduler)', 'score'] )
df_scores.sort_values('score', ascending=True)

100%|██████████| 5/5 [00:00<00:00, 21.90it/s]


,Etas,Learing Rate(Scheduler),score
8,0.100,constant,0.085431
1,0.001,optimal,0.087162
9,0.100,optimal,0.087162
5,0.010,optimal,0.087162
13,0.200,optimal,0.087162
17,0.400,optimal,0.087162
10,0.100,adaptive,0.087351
14,0.200,adaptive,0.088418
18,0.400,adaptive,0.092455
12,0.200,constant,0.093988
